# Notebook 04: Training Loop Patterns

## Why This Matters

The training loop is where theoretical understanding meets practical engineering. An incorrect training loop -- one that forgets `zero_grad`, clips gradients at the wrong point, or saves only `model.state_dict()` without the optimizer state -- can waste days of compute and produce unreproducible experiments. In production ML, the training loop is also the primary surface for LR schedule bugs, checkpoint corruption, and optimizer hyperparameter mistakes that look like model architecture problems.

## Background

The canonical gradient-descent training step is:

$$\theta_{t+1} = \theta_t - \eta \cdot \hat{g}_t$$

where $\hat{g}_t$ is the gradient estimate (potentially momentum-adjusted or adaptive). The five-step loop makes this concrete and handles all the bookkeeping PyTorch requires:

1. `optimizer.zero_grad()` -- clear accumulated gradients
2. `output = model(x)` -- forward pass
3. `loss = criterion(output, y)` -- compute scalar loss
4. `loss.backward()` -- compute gradients
5. `optimizer.step()` -- update parameters

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import math

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch: {torch.__version__}")
print("Ready.")

## 1. The Canonical 5-Step Loop

Each step has a reason for existing. Skipping or reordering any of them produces subtle bugs. The most common mistake is calling `zero_grad()` *after* `backward()` rather than before -- this discards the gradients you just computed before the optimizer can use them.

In [ ]:
# Minimal model and data for demonstration
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Fake dataset
X = torch.randn(64, 4, device=device)
y = torch.randint(0, 2, (64,), device=device)

losses = []
for step in range(5):
    # Step 1: zero gradients -- BEFORE forward, not after
    optimizer.zero_grad(set_to_none=True)   # set_to_none saves memory vs zero fill

    # Step 2: forward pass
    logits = model(X)

    # Step 3: compute loss
    loss = criterion(logits, y)

    # Step 4: backward pass -- computes gradients
    loss.backward()

    # Step 5: optimizer step -- updates parameters
    optimizer.step()

    losses.append(loss.item())
    print(f"step {step}: loss = {loss.item():.4f}")

print(f"\nLoss reduced: {losses[0]:.4f} -> {losses[-1]:.4f}")

## 2. zero_grad(set_to_none=True)

Before PyTorch 1.7 the only option was `zero_grad()`, which fills `.grad` tensors with zeros. `set_to_none=True` (the default since PyTorch 2.0) sets `.grad` to `None` instead. This is strictly better:
- Avoids allocating and writing zero-filled gradient buffers when no gradient exists yet
- Lets PyTorch skip the addition for parameters that genuinely received no gradient in the batch (sparse gradients in embeddings)
- Slightly lower memory usage because `None` takes no storage

In [ ]:
# Demonstrate the memory difference
model_a = nn.Embedding(1000, 64)
model_b = nn.Embedding(1000, 64)
for p_a, p_b in zip(model_a.parameters(), model_b.parameters()):
    p_b.data.copy_(p_a.data)  # same init

x_sparse = torch.tensor([0, 1, 2])  # only 3 of 1000 embeddings used

for model, name in [(model_a, "zero_fill"), (model_b, "set_to_none")]:
    opt = optim.SGD(model.parameters(), lr=0.01)
    model(x_sparse).sum().backward()
    if name == "zero_fill":
        opt.zero_grad(set_to_none=False)
    else:
        opt.zero_grad(set_to_none=True)
    for p in model.parameters():
        if p.grad is None:
            print(f"  {name}: grad is None (no allocation)")
        else:
            print(f"  {name}: grad is a zero tensor (allocated but unused)")

## 3. SGD vs Adam vs AdamW -- When to Use Each

Optimizer choice is one of the highest-leverage hyperparameters. The wrong optimizer can mean 2x slower convergence or worse final accuracy.

- **SGD + momentum**: Still the gold standard for image classification on fixed architectures (ResNet, ViT). Generalises better than Adam on well-tuned problems. Sensitive to LR and needs a good schedule.
- **Adam**: Adaptive per-parameter learning rates. Fast on NLP/language tasks. Tendency to overfit on some problems due to large effective LR in early training.
- **AdamW**: Adam + correct weight decay. The standard choice for transformers and NLP. Adam's weight decay was implemented incorrectly (via L2 on the gradient, not the parameter) -- AdamW fixes this. Use AdamW over Adam by default for anything transformer-based.

In [ ]:
torch.manual_seed(42)

# Synthetic noisy quadratic (simulate loss landscape)
def loss_fn(params):
    w = params[0]
    return (w - 2.0) ** 2 + 0.1 * torch.randn(1)

optimizers = {
    "SGD (mom=0.9)": (optim.SGD, {"lr": 0.1, "momentum": 0.9}),
    "Adam":           (optim.Adam,  {"lr": 0.1}),
    "AdamW":          (optim.AdamW, {"lr": 0.1, "weight_decay": 0.01}),
}

fig, ax = plt.subplots(figsize=(10, 4))
for label, (opt_cls, kwargs) in optimizers.items():
    w = torch.tensor([0.0], requires_grad=True)
    opt = opt_cls([w], **kwargs)
    history = []
    for _ in range(80):
        opt.zero_grad(set_to_none=True)
        loss = loss_fn([w])
        loss.backward()
        opt.step()
        history.append(w.item())
    ax.plot(history, label=label)

ax.axhline(2.0, color="k", linestyle="--", label="optimum (w=2)")
ax.set_xlabel("step"); ax.set_ylabel("parameter value")
ax.set_title("Optimizer Convergence on Noisy Quadratic")
ax.legend(); plt.tight_layout(); plt.savefig("/tmp/optimizer_comparison.png", dpi=80)
plt.show()
print("Optimizer comparison saved.")

## 4. Learning Rate Schedules

A constant learning rate is almost always suboptimal. The three schedules most used in production:

- **Cosine annealing**: smoothly decays LR from max to near-zero over a fixed number of steps. Good for fixed-budget training.
- **Warmup + cosine**: first linearly increase LR from 0 to max_lr over `warmup_steps`, then cosine decay. Essential for Adam-based training of transformers -- avoids early-training instability.
- **OneCycleLR**: brief warmup followed by cosine decay, with max_lr in the middle. Often finds good solutions faster than cosine alone.

In [ ]:
model_sched = nn.Linear(4, 2)
optimizer_sched = optim.AdamW(model_sched.parameters(), lr=1e-3)

total_steps = 200
warmup_steps = 20

# Manual warmup + cosine (common in transformer training)
def lr_lambda(step):
    if step < warmup_steps:
        return step / warmup_steps  # linear warmup
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))  # cosine decay

scheduler_warmup_cosine = optim.lr_scheduler.LambdaLR(optimizer_sched, lr_lambda)

# OneCycleLR
optimizer_cycle = optim.AdamW(nn.Linear(4,2).parameters(), lr=1e-5)
scheduler_cycle = optim.lr_scheduler.OneCycleLR(
    optimizer_cycle, max_lr=1e-3, total_steps=total_steps
)

# CosineAnnealingLR
optimizer_cos = optim.SGD(nn.Linear(4,2).parameters(), lr=0.1)
scheduler_cos = optim.lr_scheduler.CosineAnnealingLR(optimizer_cos, T_max=total_steps)

fig, ax = plt.subplots(figsize=(10, 4))
lr_warmup_cos, lr_cycle, lr_cosine = [], [], []
for _ in range(total_steps):
    lr_warmup_cos.append(scheduler_warmup_cosine.get_last_lr()[0])
    lr_cycle.append(scheduler_cycle.get_last_lr()[0])
    lr_cosine.append(scheduler_cos.get_last_lr()[0])
    scheduler_warmup_cosine.step()
    scheduler_cycle.step()
    scheduler_cos.step()

ax.plot(lr_warmup_cos, label="Warmup + Cosine (AdamW)")
ax.plot(lr_cycle,      label="OneCycleLR")
ax.plot(lr_cosine,     label="CosineAnnealingLR (SGD)")
ax.set_xlabel("step"); ax.set_ylabel("learning rate")
ax.set_title("Learning Rate Schedules"); ax.legend()
plt.tight_layout(); plt.savefig("/tmp/lr_schedules.png", dpi=80); plt.show()
print("LR schedule plot saved.")

## 5. Gradient Clipping

Gradient clipping prevents a single bad batch from taking a catastrophically large parameter update. There are two variants:

- `clip_grad_norm_(params, max_norm)` -- scales the global gradient vector so its L2 norm does not exceed `max_norm`. This is the standard for transformers and RNNs. It preserves gradient direction while limiting magnitude.
- `clip_grad_value_(params, clip_value)` -- clamps each gradient element independently. Less common; can distort gradient direction.

**Critical ordering**: if you use a GradScaler (mixed precision), you must call `scaler.unscale_(optimizer)` BEFORE clipping, so clipping operates on real-scale gradients, not scaled ones.

In [ ]:
torch.manual_seed(0)
model_clip = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 2)).to(device)
optimizer_clip = optim.AdamW(model_clip.parameters(), lr=1e-3)

# Create a batch that produces large gradients
X_big = torch.randn(8, 4, device=device) * 100  # scaled-up inputs
y_big = torch.randint(0, 2, (8,), device=device)
criterion = nn.CrossEntropyLoss()

optimizer_clip.zero_grad(set_to_none=True)
loss = criterion(model_clip(X_big), y_big)
loss.backward()

# Check grad norm before clipping
total_norm_before = 0.0
for p in model_clip.parameters():
    if p.grad is not None:
        total_norm_before += p.grad.data.norm(2).item() ** 2
total_norm_before = total_norm_before ** 0.5
print(f"Gradient norm BEFORE clipping: {total_norm_before:.2f}")

# Clip to max_norm=1.0
nn.utils.clip_grad_norm_(model_clip.parameters(), max_norm=1.0)

total_norm_after = 0.0
for p in model_clip.parameters():
    if p.grad is not None:
        total_norm_after += p.grad.data.norm(2).item() ** 2
total_norm_after = total_norm_after ** 0.5
print(f"Gradient norm AFTER  clipping: {total_norm_after:.4f}  (max_norm=1.0)")

optimizer_clip.step()
print("Step complete.")

## 6. Checkpointing -- Save Model + Optimizer + Scheduler + Epoch

A checkpoint that only saves `model.state_dict()` cannot resume training correctly. The optimizer state (momentum buffers, Adam's m and v moments) takes as much memory as the model itself and encodes training history that is critical for convergence. Losing it on resume restarts the optimizer warm-up. Always save and restore the full training state.

In [ ]:
import os, tempfile

# Setup
torch.manual_seed(42)
model_ckpt = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2)).to(device)
optimizer_ckpt = optim.AdamW(model_ckpt.parameters(), lr=1e-3)
scheduler_ckpt = optim.lr_scheduler.CosineAnnealingLR(optimizer_ckpt, T_max=100)

# Simulate 10 steps of training
X_ck = torch.randn(16, 4, device=device)
y_ck = torch.randint(0, 2, (16,), device=device)
for epoch in range(10):
    optimizer_ckpt.zero_grad(set_to_none=True)
    loss = nn.CrossEntropyLoss()(model_ckpt(X_ck), y_ck)
    loss.backward()
    nn.utils.clip_grad_norm_(model_ckpt.parameters(), max_norm=1.0)
    optimizer_ckpt.step()
    scheduler_ckpt.step()

# Save full checkpoint
ckpt = {
    "epoch":           10,
    "model_state":     model_ckpt.state_dict(),
    "optimizer_state": optimizer_ckpt.state_dict(),
    "scheduler_state": scheduler_ckpt.state_dict(),
    "loss":            loss.item(),
}
ckpt_path = "/tmp/demo_checkpoint.pt"
torch.save(ckpt, ckpt_path)
print(f"Checkpoint saved: {os.path.getsize(ckpt_path)} bytes")

# Resume: create fresh objects and restore
model_resumed = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2)).to(device)
optimizer_resumed = optim.AdamW(model_resumed.parameters(), lr=1e-3)
scheduler_resumed = optim.lr_scheduler.CosineAnnealingLR(optimizer_resumed, T_max=100)

loaded = torch.load(ckpt_path, map_location=device)
model_resumed.load_state_dict(loaded["model_state"])
optimizer_resumed.load_state_dict(loaded["optimizer_state"])
scheduler_resumed.load_state_dict(loaded["scheduler_state"])
start_epoch = loaded["epoch"]

print(f"Resumed from epoch {start_epoch}")
print(f"LR after resume: {scheduler_resumed.get_last_lr()[0]:.6f}")
print("Checkpoint keys:", list(loaded.keys()))

## Summary

### Key Concepts

| Concept | Key Point |
|---------|----------|
| 5-step loop | zero_grad -> forward -> loss -> backward -> step; order is non-negotiable |
| set_to_none=True | Saves memory vs zero-filling; default in PyTorch 2.0+ |
| SGD vs AdamW | SGD+mom for vision; AdamW for transformers and NLP |
| Warmup + cosine | Linear warmup prevents early AdamW instability; cosine for smooth decay |
| clip_grad_norm_ | Scales gradient vector; preserves direction; clip AFTER unscale_ with GradScaler |
| Full checkpoint | Save model + optimizer + scheduler + epoch; optimizer state is as critical as model state |

### Common Pitfalls
- Calling `zero_grad()` AFTER `backward()` -- you zero the gradients you just computed
- Not including warmup with AdamW -- causes loss spikes or divergence in early training
- Using `clip_grad_value_` when you meant `clip_grad_norm_` -- distorts gradient direction
- Saving only `model.state_dict()` -- resume restarts optimizer; LR schedule resets
- Not calling `scheduler.step()` -- LR stays at initial value for the entire run

### Next: Notebook 05 -- DataLoader and Dataset